In [ ]:

#@title ✅ 환경 설정 (Colab 권장)
# (Colab) 런타임 유형: GPU
# !nvidia-smi

# 필수 패키지 설치
# 최신 버전이 필요하면 주석 해제 후 실행하세요.
# %pip install -U diffusers==0.31.0 transformers==4.44.2 accelerate==0.34.2 safetensors==0.4.5
# %pip install -U invisible-watermark sentencepiece
# xFormers는 옵션이지만, 메모리/속도 측면에서 유리합니다 (Colab GPU 환경에 따라 설치 실패할 수 있음).
# %pip install -U xformers

import torch, os, math, itertools, time
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())


In [ ]:

#@title 🔧 텍스트 임베딩 추출 유틸
from typing import List, Tuple

def get_text_embeddings(pipe: StableDiffusionPipeline, prompts: List[str]):
    """
    pipe.tokenizer & pipe.text_encoder를 사용하여 텍스트 임베딩을 구합니다.
    반환: (prompt_embeds, negative_embeds)
    - prompt_embeds: [B, 77, hidden]
    - negative_embeds: [B, 77, hidden]  (빈 문자열 negative prompt)
    """
    text_inputs = pipe.tokenizer(
        prompts,
        padding="max_length",
        max_length=pipe.tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    )
    with torch.no_grad():
        prompt_embeds = pipe.text_encoder(text_inputs.input_ids.to(pipe.device))[0]
        # 기본 negative prompt을 빈 문자열로 동일 배치 크기만큼 생성
        uncond_input = pipe.tokenizer(
            [""] * len(prompts),
            padding="max_length",
            max_length=pipe.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt",
        )
        negative_embeds = pipe.text_encoder(uncond_input.input_ids.to(pipe.device))[0]
    return prompt_embeds, negative_embeds


In [ ]:

#@title 📌 프롬프트 4개를 지정하세요
prompt_1 = "A watercolor painting of a Golden Retriever at the beach" #@param {type:"string"}
prompt_2 = "A still life DSLR photo of a bowl of fruit"               #@param {type:"string"}
prompt_3 = "The Eiffel Tower in the style of Starry Night"            #@param {type:"string"}
prompt_4 = "An architectural sketch of a skyscraper"                   #@param {type:"string"}

prompts = [prompt_1, prompt_2, prompt_3, prompt_4]
print(prompts)


In [ ]:

#@title 🧩 Stable Diffusion 파이프라인 준비
# 베이스 모델 선택 (1.x 계열은 속도/메모리 친화적)
model_id = "CompVis/stable-diffusion-v1-4"  #@param ["CompVis/stable-diffusion-v1-4", "runwayml/stable-diffusion-v1-5", "stabilityai/stable-diffusion-2-1"]
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# 옵션: 메모리 최적화
pipe.enable_attention_slicing()
if device == "cuda":
    pipe.enable_vae_slicing()
    # pipe.enable_model_cpu_offload()  # GPU 메모리 부족할 때
print("Loaded:", model_id)


In [ ]:

#@title 🔢 선형 보간 함수 (slerp 옵션 포함)
def slerp(t, v0, v1, eps=1e-8):
    """Spherical linear interpolation for better manifold traversal (optional)."""
    v0_u = F.normalize(v0, dim=-1)
    v1_u = F.normalize(v1, dim=-1)
    dot = (v0_u * v1_u).sum(dim=-1, keepdim=True).clamp(-1+eps, 1-eps)
    omega = torch.acos(dot)
    so = torch.sin(omega)
    return (torch.sin((1.0 - t) * omega) / so) * v0 + (torch.sin(t * omega) / so) * v1

def interpolate_embeddings(e0, e1, num_steps=5, mode="linear"):
    """
    e0, e1: [1, 77, hidden] 텍스트 임베딩
    num_steps: 중간 단계 수 (끝점 포함 X). 출력은 길이 num_steps+2 (시작/끝 포함)
    mode: "linear" | "slerp"
    """
    outs = []
    for i in range(num_steps + 2):
        t = i / (num_steps + 1)
        if mode == "slerp":
            emb = slerp(torch.tensor(t, device=e0.device), e0, e1)
        else:
            emb = (1 - t) * e0 + t * e1
        outs.append(emb)
    return outs


In [ ]:

#@title 🎨 보간 경로에서 이미지 생성
pair = "p1→p2"  #@param ["p1→p2", "p3→p4", "p1→p3", "p2→p4"]
steps_between = 5  #@param {type:"slider", min:2, max:10, step:1}
guidance_scale = 7.5  #@param {type:"slider", min:1.0, max:15.0, step:0.5}
num_inference_steps = 30  #@param {type:"slider", min:10, max:75, step:5}
interp_mode = "linear" #@param ["linear", "slerp"]
seed = 42  #@param {type:"integer"}

torch.manual_seed(seed)

# 선택된 페어 매핑
idx_map = {
    "p1→p2": (0, 1),
    "p3→p4": (2, 3),
    "p1→p3": (0, 2),
    "p2→p4": (1, 3),
}
i0, i1 = idx_map[pair]

# 임베딩 추출
with torch.no_grad():
    prompt_embeds_all, neg_embeds_all = get_text_embeddings(pipe, prompts)

# 시작/끝 임베딩 선택
e0 = prompt_embeds_all[i0:i0+1]
e1 = prompt_embeds_all[i1:i1+1]
neg = neg_embeds_all[i0:i0+1]  # negative는 아무거나 1개면 충분

# 보간 임베딩들
emb_list = interpolate_embeddings(e0, e1, num_steps=steps_between, mode=interp_mode)

# 이미지 생성
images = []
for idx, emb in enumerate(emb_list):
    img = pipe(
        prompt_embeds=emb,
        negative_prompt_embeds=neg,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
    ).images[0]
    images.append(img)

# 시각화
cols = len(images)
plt.figure(figsize=(1.8*cols, 3))
for i, img in enumerate(images):
    plt.subplot(1, cols, i+1)
    plt.imshow(img)
    plt.axis('off')
plt.suptitle(f"{pair} — {interp_mode} interpolation — g={guidance_scale}, steps={num_inference_steps}")
plt.show()


In [ ]:

#@title ⚖️ Guidance scale sweep
test_prompt = "A cinematic photo of a husky running in the snow, golden hour" #@param {type:"string"}
scales = "3,5,7.5,10,12"  #@param {type:"string"}
num_inference_steps = 30  #@param {type:"slider", min:10, max:75, step:5}
seed = 123  #@param {type:"integer"}

scales = [float(x.strip()) for x in scales.split(",")]
torch.manual_seed(seed)

with torch.no_grad():
    p_emb, n_emb = get_text_embeddings(pipe, [test_prompt])

imgs = []
for s in scales:
    img = pipe(
        prompt_embeds=p_emb,
        negative_prompt_embeds=n_emb,
        guidance_scale=s,
        num_inference_steps=num_inference_steps,
    ).images[0]
    imgs.append(img)

plt.figure(figsize=(2.4*len(scales), 3))
for i, (s, img) in enumerate(zip(scales, imgs)):
    plt.subplot(1, len(scales), i+1)
    plt.imshow(img)
    plt.title(f"g={s}")
    plt.axis('off')
plt.suptitle("Guidance scale effect")
plt.show()


In [ ]:
from huggingface_hub import login
login()  # 브라우저에서 토큰 발급 후 입력

In [ ]:
#@title ✅ DreamBooth 실행 (diffusers main로 업그레이드 + logger 옵션 제거)
import os, sys, subprocess, textwrap, urllib.request
from pathlib import Path

# 0) diffusers를 소스(main)로 업그레이드 → check_min_version 통과
!pip -q uninstall -y diffusers
!pip -q install --no-cache-dir -U git+https://github.com/huggingface/diffusers.git@main
!pip -q install -U accelerate transformers safetensors invisible-watermark sentencepiece

import torch, diffusers, accelerate, transformers  # noqa
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("diffusers:", diffusers.__version__)

# 1) DreamBooth 스크립트 다운로드(항상 main 사용: 업그레이드와 버전 일치)
EX_DIR = Path("/content/diffusers_examples/dreambooth"); EX_DIR.mkdir(parents=True, exist_ok=True)
SCRIPT = EX_DIR / "train_dreambooth.py"
url = "https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth.py"
urllib.request.urlretrieve(url, SCRIPT)
print("Script path:", SCRIPT, "| Exists:", SCRIPT.exists())

# 2) 데이터/출력 경로
INSTANCE_DIR = Path("/content/data/instance")
CLASS_DIR    = Path("/content/data/class")
OUTPUT_DIR   = Path("/content/dreambooth_out")
for p in [INSTANCE_DIR, CLASS_DIR, OUTPUT_DIR]: p.mkdir(parents=True, exist_ok=True)

def count_imgs(p: Path):
    exts = {".jpg",".jpeg",".png",".webp",".bmp",".gif"}
    return sum(1 for f in p.glob("**/*") if f.suffix.lower() in exts)

print(f"Instance images: {count_imgs(INSTANCE_DIR)} at {INSTANCE_DIR}")
print(f"Class images   : {count_imgs(CLASS_DIR)} at {CLASS_DIR}")

# 3) Accelerate 무인 설정
acc_dir = Path.home()/".cache/huggingface/accelerate/default_config"
acc_dir.mkdir(parents=True, exist_ok=True)
cfg = acc_dir/"config.yaml"
if not cfg.exists():
    cfg.write_text(textwrap.dedent("""\
        compute_environment: LOCAL_MACHINE
        distributed_type: NO
        mixed_precision: fp16
        use_cpu: false
        num_processes: 1
        machine_rank: 0
        gpu_ids: "0"
        rdzv_backend: static
        same_network: true
        main_training_function: main
        deepspeed_config: {}
    """))

# 4) 하이퍼파라미터
BASE_MODEL         = "CompVis/stable-diffusion-v1-4"
INSTANCE_PROMPT    = "a photo of sks-celeb person"
CLASS_PROMPT       = "a photo of a person"
RESOLUTION         = 512
TRAIN_BATCH_SIZE   = 1
MAX_TRAIN_STEPS    = 400
NUM_CLASS_IMAGES   = 100
PRIOR_LOSS_WEIGHT  = 1.0

# 5) 실행 커맨드 (logger 옵션 제거)
cmd = [
    sys.executable, "-m", "accelerate.commands.launch",
    "--num_processes", "1", "--num_machines", "1",
    "--mixed_precision", "fp16", "--dynamo_backend", "no",
    str(SCRIPT),
    f"--pretrained_model_name_or_path={BASE_MODEL}",
    f"--instance_data_dir={str(INSTANCE_DIR)}",
    f"--class_data_dir={str(CLASS_DIR)}",
    f"--output_dir={str(OUTPUT_DIR)}",
    f"--instance_prompt={INSTANCE_PROMPT}",
    f"--class_prompt={CLASS_PROMPT}",
    f"--resolution={RESOLUTION}",
    f"--train_batch_size={TRAIN_BATCH_SIZE}",
    "--gradient_accumulation_steps=1",
    "--gradient_checkpointing",
    "--learning_rate=2e-6",
    "--lr_scheduler=constant",
    "--lr_warmup_steps=0",
    f"--num_class_images={NUM_CLASS_IMAGES}",
    f"--max_train_steps={MAX_TRAIN_STEPS}",
    "--with_prior_preservation",
    f"--prior_loss_weight={PRIOR_LOSS_WEIGHT}",
    "--dataloader_num_workers=0",
    "--checkpointing_steps=50",
    "--checkpoints_total_limit=3",
    # ← --report_to 옵션 제거 (none 때문에 에러 났었음)
]

print("=== Launch ===\n", " ".join(cmd), "\n")
log_path = "/content/dreambooth_train.log"
print(f"Logging to {log_path}")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with open(log_path, "w") as lf:
    for line in proc.stdout:
        print(line, end="")
        lf.write(line)
proc.wait()
print("\nReturn code:", proc.returncode)
if proc.returncode != 0:
    print("❌ 실패. 마지막 수십 줄 로그를 확인하세요 →", log_path)
else:
    print("✅ 성공. 가중치가 저장되었습니다 →", OUTPUT_DIR)


In [ ]:

#@title 🧩 체크포인트 + LoRA 로딩 및 생성
from diffusers import AutoPipelineForText2Image

base_model_id = "digiplay/hellofantasytime_v1.22"  #@param ["digiplay/hellofantasytime_v1.22", "CompVis/stable-diffusion-v1-4", "runwayml/stable-diffusion-v1-5"]
lora_path = "/content/lora_weight.safetensors"     #@param {type:"string"}
prompt = "A fantasy landscape with floating islands, ultra-detailed, volumetric lighting" #@param {type:"string"}
negative_prompt = "" #@param {type:"string"}
guidance_scale = 7.0 #@param {type:"slider", min:1.0, max:15.0, step:0.5}
num_inference_steps = 30 #@param {type:"slider", min:10, max:75, step:5}
seed = 1234 #@param {type:"integer"}

pipe_lo = AutoPipelineForText2Image.from_pretrained(
    base_model_id, torch_dtype=torch.float16 if device=="cuda" else torch.float32
).to(device)
pipe_lo.scheduler = DPMSolverMultistepScheduler.from_config(pipe_lo.scheduler.config)

# LoRA 로드
try:
    pipe_lo.load_lora_weights(lora_path)
    print("Loaded LoRA:", lora_path)
except Exception as e:
    print("LoRA 로드 실패 (경로 확인):", e)

# 생성
generator = torch.Generator(device=device).manual_seed(seed)
image = pipe_lo(
    prompt=prompt,
    negative_prompt=negative_prompt if negative_prompt else None,
    guidance_scale=guidance_scale,
    num_inference_steps=num_inference_steps,
    generator=generator
).images[0]

display(image)


In [ ]:
#@title 🖼️ DreamBooth ckpt-400 병합 후 생성 (A100 최적화 포함)
import os, torch
from pathlib import Path
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

# ===== 설정 =====
device = "cuda" if torch.cuda.is_available() else "cpu"
base_model_id = "CompVis/stable-diffusion-v1-4"                 # DreamBooth 베이스
ckpt_dir      = Path("/content/dreambooth_out/checkpoint-400")  # 사용할 체크포인트
merged_dir    = Path("/content/dreambooth_out_merged")          # 병합 결과 저장 폴더

prompt = "a portrait photo of sks-celeb person, **looking off-camera**, thoughtful mood, **hand in pocket**, soft key light, hair light, 85mm, f1.4, bokeh"
negative_prompt = "female, woman"   # 필요 시 "female, woman" 등 최소한으로
guidance_scale = 8.0
num_inference_steps = 30
seed = 711

# ===== 0) 기존에 병합된 파이프라인이 있으면 바로 사용 =====
if (merged_dir / "model_index.json").exists():
    print(f"Using existing merged pipeline: {merged_dir}")
    pipe = StableDiffusionPipeline.from_pretrained(
        str(merged_dir),
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device)
else:
    # ===== 1) 베이스 파이프라인 로드 =====
    print("Merging checkpoint weights into base model…")
    pipe = StableDiffusionPipeline.from_pretrained(
        base_model_id,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device)

    # ===== 2) ckpt-400에서 가중치 주입 (UNet 필수, Text Encoder는 있으면 함께) =====
    def load_state(module, candidates):
        import safetensors.torch as sft
        for p in candidates:
            p = Path(p)
            if p.exists():
                # safetensors 우선, 그 외 bin/pt 허용
                sd = sft.load_file(str(p)) if p.suffix == ".safetensors" else torch.load(str(p), map_location="cpu")
                missing, unexpected = module.load_state_dict(sd, strict=False)
                print(f"Loaded {p}  (missing={len(missing)}, unexpected={len(unexpected)})")
                return True
        return False

    ok_unet = load_state(pipe.unet, [
        ckpt_dir / "unet/diffusion_pytorch_model.safetensors",
        ckpt_dir / "unet/diffusion_pytorch_model.bin",
        ckpt_dir / "unet/pytorch_model.bin",
    ])
    # (선택) 텍스트 인코더도 저장돼 있으면 함께 로드
    _ = load_state(pipe.text_encoder, [
        ckpt_dir / "text_encoder/model.safetensors",
        ckpt_dir / "text_encoder/pytorch_model.bin",
    ])

    if not ok_unet:
        raise SystemExit("❌ ckpt-400에서 UNet 가중치를 찾지 못했습니다. 폴더 구조/파일명을 확인하세요.")

    # ===== 3) 병합된 파이프라인 저장 → 다음부터는 바로 로드 가능 =====
    merged_dir.mkdir(parents=True, exist_ok=True)
    pipe.save_pretrained(str(merged_dir))
    print(f"✅ Saved merged pipeline to: {merged_dir}")

# ===== 4) A100 최적화 & 스케줄러 =====
try:
    pipe.enable_sdpa()  # PyTorch 2.x SDPA (A100에서 빠르고 메모리 효율적)
except Exception:
    pass

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# ===== 5) 생성 =====
generator = torch.Generator(device=device).manual_seed(seed)
kwargs = dict(
    prompt=prompt,
    guidance_scale=guidance_scale,
    num_inference_steps=num_inference_steps,
    generator=generator,
)
if negative_prompt.strip():
    kwargs["negative_prompt"] = negative_prompt

image = pipe(**kwargs).images[0]
display(image)


## 회고
```
instance 사진을 적게 둬서 그런지 실제 인물과 생성한 이미지의 인물이 전혀 닮아보이지 않고 seed 값을 다르게 주면 instance 에 넣은 남자(차은우)인물이 아닌 여자 인물 사진이 생성되는 경우도 있었다. 이를 위해 instance 사진을 늘리거나 class 사진에 남자 및 여자의 데이터 비율도 확인하여 어떤 요인을 변경해야 모델 성능(눈으로 봤을 때 instance 인물과 비슷한지)이 높아질지 실험이 필요하다.
```